
# people_segment.csv → persona_attributes_weighted.jsonl (v4: meta 포함)

**cluster / label / Description** 를 `meta`로 함께 포함해, 이후 프롬프트 생성(B/C) 단계에서
클러스터 컨텍스트로 활용할 수 있도록 업데이트한 버전입니다.


In [ ]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

INPUT_CSV = Path("people_segment.csv")
OUT_JSONL = Path("persona_attributes_weighted.jsonl")
OUT_PREVIEW = Path("persona_attributes_weighted_preview.json")

# 가중치 배분 비율(합=1 권장)
DEMO_SHARE = 0.30
BEHAV_SHARE = 0.70

# (선택) 인구통계 컬럼 직접 지정 (지정 시 자동탐지 무시)
DEMO_COLS_OVERRIDE = []
# 예시 DEMO_COLS_OVERRIDE = ["성별", "연령", "소득", "직업", "지역", "가족구성"]

# 인구통계 내부 항목별 가중
DEMO_FIELD_WEIGHTS = {}
# 예시 DEMO_FIELD_WEIGHTS = { "연령": 2.0, "성별": 1.0, "소득": 1.0 }

# 성향 내부 multiplier (키는 *_scaled)
BEHAV_FIELD_MULTIPLIERS = {}
# 예시 BEHAV_FIELD_MULTIPLIERS = { "brand_loyalty_scaled": 1.2, "price_sensitivity_scaled": 0.9 }

# meta 필드로 고려할 칼럼명(있으면 자동으로 meta에 포함)
META_KEYS = ["cluster","label","Description"]

print("CONFIG loaded.")

CONFIG loaded.


In [15]:

# =============================
# 1) Load & clean
# =============================
import pandas as pd, re, numpy as np

def read_csv_fallback(path: Path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")

df = read_csv_fallback(INPUT_CSV)

def clean_colname(c: str) -> str:
    return re.sub(r"'([^']+)'", r"\1", c)

df.rename(columns={c: clean_colname(c) for c in df.columns}, inplace=True)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head(3)

Rows: 363
Columns: 20


,id,gender,age,job,education,region,household,marriage,income_status,income_month,cluster,brand_loyalty_scaled,cooking_convenience_scaled,health_orientation_scaled,hmr_preference_scaled,premium_orientation_scaled,price_sensitivity_scaled,variety_seeking_scaled,label,Description
0,1,남자,30대,사무 종사자,대학교 졸업(전문대졸/대학원생 포함),경상북도,1세대가족,기혼,맞벌이 함,500-600만원 미만,0,0.208955,0.429538,0.534351,0.698335,0.286645,0.229087,0.526087,실속형 미식가,편리성을 중시하면서도 새로운 맛과 제품을 시도하는 데 적극적인 소비자 그룹입니다. ...
1,2,남자,30대,사무 종사자,대학교 졸업(전문대졸/대학원생 포함),서울특별시,1세대가족,기혼,맞벌이 함,600-700만원 미만,1,0.119403,0.642991,0.668484,0.902057,0.355700,0.402638,0.421739,건강 추구형 소비자,가격이나 브랜드에 크게 구애받지 않고 건강을 최우선으로 고려하는 프리미엄 소비자 그...
2,3,남자,30대,사무 종사자,대학교 졸업(전문대졸/대학원생 포함),세종특별자치시,1세대가족,기혼,맞벌이 함,400-500만원 미만,3,0.852985,0.868470,0.102236,0.902057,0.375570,0.487331,0.321739,트렌드 주도형 소비자,자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가...


In [16]:

# =============================
# 2) Auto-detect demographics & behavioral columns
# =============================
# behavior: *_scaled
behav_cols = [c for c in df.columns if c.endswith("_scaled")]

# demographics auto-detect
blacklist = {"id","cluster","segment_id","label","Description","Unnamed: 0"}
cat_candidates = []
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if df[c].dtype == "object":
        nun = df[c].nunique(dropna=False)
        ratio = nun / max(1, len(df))
        if ratio <= 0.5:
            cat_candidates.append(c)

# numeric small-cardinality also considered
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if np.issubdtype(df[c].dtype, np.number):
        nun = df[c].nunique(dropna=False)
        if 2 <= nun <= 20:
            cat_candidates.append(c)

if isinstance(DEMO_COLS_OVERRIDE, (list, tuple)) and len(DEMO_COLS_OVERRIDE) > 0:
    demo_cols = [c for c in DEMO_COLS_OVERRIDE if c in df.columns]
else:
    seen = set(); demo_cols = []
    for c in cat_candidates:
        if c not in seen:
            seen.add(c); demo_cols.append(c)
        if len(demo_cols) >= 7: break

if not behav_cols:
    raise RuntimeError("No behavioral (_scaled) columns detected.")

id_col = "id" if "id" in df.columns else None
print("Demographics:", demo_cols)
print("Behavioral:", behav_cols)
print("ID col:", id_col)

Demographics: ['gender', 'age', 'job', 'education', 'region', 'household', 'marriage']
Behavioral: ['brand_loyalty_scaled', 'cooking_convenience_scaled', 'health_orientation_scaled', 'hmr_preference_scaled', 'premium_orientation_scaled', 'price_sensitivity_scaled', 'variety_seeking_scaled']
ID col: id


In [17]:

# =============================
# 3) Helpers
# =============================
from typing import Dict, Any

def normalize_weights(raw: Dict[str, float], target_sum: float) -> Dict[str, float]:
    total = sum(v for v in raw.values() if pd.notna(v))
    if total <= 0:
        n = len(raw)
        return {k: (target_sum / n if n else 0.0) for k in raw}
    return {k: (v / total) * target_sum for k, v in raw.items()}

def build_persona_attributes(row: pd.Series,
                             demo_cols, behav_cols,
                             demo_share: float, behav_share: float,
                             demo_field_weights: Dict[str, float],
                             behav_field_multipliers: Dict[str, float]) -> Dict[str, Any]:
    demo_raw = {c: demo_field_weights.get(c, 1.0) for c in demo_cols}
    demo_weights = normalize_weights(demo_raw, demo_share)
    demo_attrs = {c: {"value": (None if pd.isna(row.get(c)) else row.get(c)),
                      "weight": float(demo_weights.get(c, 0.0))}
                  for c in demo_cols}

    behav_raw = {}
    for c in behav_cols:
        val = row.get(c); 
        if pd.isna(val): val = 0.0
        mult = behav_field_multipliers.get(c, 1.0)
        behav_raw[c] = float(val) * float(mult)
    behav_weights = normalize_weights(behav_raw, behav_share)
    behav_attrs = {c: {"value": float(row.get(c, 0.0)),
                       "weight": float(behav_weights.get(c, 0.0))}
                   for c in behav_cols}

    attributes = {**demo_attrs, **behav_attrs}
    total_weight = sum(v["weight"] for v in attributes.values())
    if total_weight > 0:
        for k in attributes:
            attributes[k]["weight"] = attributes[k]["weight"] / total_weight
    return attributes

In [18]:

# =============================
# 4) Build & save with meta
# =============================
import json

total_share = DEMO_SHARE + BEHAV_SHARE
if abs(total_share - 1.0) > 1e-8:
    print(f"[WARN] DEMO_SHARE + BEHAV_SHARE = {total_share:.4f}. 자동 정규화합니다.")
    DEMO_SHARE = DEMO_SHARE / total_share
    BEHAV_SHARE = BEHAV_SHARE / total_share
    print(f" -> DEMO_SHARE={DEMO_SHARE:.4f}, BEHAV_SHARE={BEHAV_SHARE:.4f}")

records = []
for idx, row in df.iterrows():
    key = row.get("id", f"row_{idx}")
    attrs = build_persona_attributes(row, demo_cols, behav_cols,
                                     DEMO_SHARE, BEHAV_SHARE,
                                     DEMO_FIELD_WEIGHTS, BEHAV_FIELD_MULTIPLIERS)
    meta = {}
    for k in META_KEYS:
        if k in df.columns:
            v = row.get(k)
            if pd.isna(v):
                v = None
            meta[k] = v

    rec = {"persona_key": key, "attributes": attrs, "meta": meta}
    records.append(rec)

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved -> {OUT_JSONL} (records={len(records)})")

# preview
with OUT_PREVIEW.open("w", encoding="utf-8") as f:
    f.write(json.dumps(records[:3], ensure_ascii=False, indent=2))
print(f"Saved preview -> {OUT_PREVIEW}")
records[:1]

Saved -> persona_attributes_weighted.jsonl (records=363)
Saved preview -> persona_attributes_weighted_preview.json


[{'persona_key': 1,
  'attributes': {'gender': {'value': '남자', 'weight': 0.04285714285714286},
   'age': {'value': '30대', 'weight': 0.04285714285714286},
   'job': {'value': '사무 종사자', 'weight': 0.04285714285714286},
   'education': {'value': '대학교 졸업(전문대졸/대학원생 포함)',
    'weight': 0.04285714285714286},
   'region': {'value': '경상북도', 'weight': 0.04285714285714286},
   'household': {'value': '1세대가족', 'weight': 0.04285714285714286},
   'marriage': {'value': '기혼', 'weight': 0.04285714285714286},
   'brand_loyalty_scaled': {'value': 0.208955223880597,
    'weight': 0.05021241005282669},
   'cooking_convenience_scaled': {'value': 0.4295377677564825,
    'weight': 0.10321889124001504},
   'health_orientation_scaled': {'value': 0.5343511450381679,
    'weight': 0.1284057814328011},
   'hmr_preference_scaled': {'value': 0.6983349657198823,
    'weight': 0.167811462196277},
   'premium_orientation_scaled': {'value': 0.2866449511400651,
    'weight': 0.06888142616832624},
   'price_sensitivity_scal